# Retail Sales Analysis with Pandas

Table-based analysis of 10,000 retail orders. No charts or plotting libraries are used.

## 1. Import pandas and load the CSV

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

sales = pd.read_csv("datasets/01_retail_sales.csv")
sales.head()

## 2. Inspect shape, columns, and data types

In [ ]:
print(f"Rows: {sales.shape[0]:,}")
print(f"Columns: {sales.shape[1]}")
pd.DataFrame({
    "column": sales.columns,
    "data_type": sales.dtypes.astype(str).values,
    "unique_values": sales.nunique().values,
})

## 3. Check data quality

In [ ]:
quality_check = pd.DataFrame({
    "missing_values": sales.isna().sum(),
    "missing_pct": sales.isna().mean().mul(100),
    "unique_values": sales.nunique(),
})
print(f"Duplicate rows: {sales.duplicated().sum()}")
print(f"Duplicate order IDs: {sales['order_id'].duplicated().sum()}")
quality_check

## 4. Prepare analytical fields

In [ ]:
sales["order_date"] = pd.to_datetime(sales["order_date"])
sales["year"] = sales["order_date"].dt.year
sales["month"] = sales["order_date"].dt.to_period("M")
sales["profit_margin_pct"] = sales["profit"].div(sales["revenue"]).mul(100)
sales["is_returned"] = sales["returned"].eq("Yes")
sales[["order_date", "year", "month", "profit_margin_pct", "is_returned"]].head()

## 5. Summarize numerical columns

In [ ]:
numeric_columns = [
    "units_sold", "unit_price", "discount_pct",
    "revenue", "cost", "profit", "profit_margin_pct"
]
sales[numeric_columns].describe().T

## 6. Calculate overall KPIs

In [ ]:
overall_kpis = pd.Series({
    "Total Orders": sales["order_id"].nunique(),
    "Total Units Sold": sales["units_sold"].sum(),
    "Total Revenue": sales["revenue"].sum(),
    "Total Cost": sales["cost"].sum(),
    "Total Profit": sales["profit"].sum(),
    "Average Order Value": sales["revenue"].mean(),
    "Overall Profit Margin %": sales["profit"].sum() / sales["revenue"].sum() * 100,
    "Return Rate %": sales["is_returned"].mean() * 100,
})
overall_kpis.to_frame("value")

## 7. Analyze monthly sales and growth

In [ ]:
monthly_sales = sales.groupby("month").agg(
    orders=("order_id", "nunique"),
    units_sold=("units_sold", "sum"),
    revenue=("revenue", "sum"),
    profit=("profit", "sum"),
)
monthly_sales["revenue_growth_pct"] = monthly_sales["revenue"].pct_change().mul(100)
monthly_sales

## 8. Compare product categories

In [ ]:
category_analysis = sales.groupby("category").agg(
    orders=("order_id", "nunique"),
    units_sold=("units_sold", "sum"),
    revenue=("revenue", "sum"),
    profit=("profit", "sum"),
    average_discount_pct=("discount_pct", "mean"),
    return_rate_pct=("is_returned", lambda values: values.mean() * 100),
).sort_values("revenue", ascending=False)
category_analysis["revenue_share_pct"] = category_analysis["revenue"].div(sales["revenue"].sum()).mul(100)
category_analysis["profit_margin_pct"] = category_analysis["profit"].div(category_analysis["revenue"]).mul(100)
category_analysis

## 9. Find the top products

In [ ]:
product_analysis = sales.groupby(["category", "product"]).agg(
    orders=("order_id", "nunique"),
    units_sold=("units_sold", "sum"),
    revenue=("revenue", "sum"),
    profit=("profit", "sum"),
).sort_values("revenue", ascending=False)
product_analysis.head(10)

## 10. Compare regional performance

In [ ]:
region_analysis = sales.groupby("region").agg(
    orders=("order_id", "nunique"),
    revenue=("revenue", "sum"),
    profit=("profit", "sum"),
    average_order_value=("revenue", "mean"),
    return_rate_pct=("is_returned", lambda values: values.mean() * 100),
).sort_values("revenue", ascending=False)
region_analysis["profit_margin_pct"] = region_analysis["profit"].div(region_analysis["revenue"]).mul(100)
region_analysis

## 11. Compare sales channels and customer segments

In [ ]:
channel_analysis = sales.groupby("sales_channel").agg(
    orders=("order_id", "nunique"),
    revenue=("revenue", "sum"),
    profit=("profit", "sum"),
    average_order_value=("revenue", "mean"),
).sort_values("revenue", ascending=False)

segment_analysis = sales.groupby("customer_segment").agg(
    orders=("order_id", "nunique"),
    units_sold=("units_sold", "sum"),
    revenue=("revenue", "sum"),
    profit=("profit", "sum"),
).sort_values("revenue", ascending=False)

print("Sales channel performance")
print(channel_analysis)
print("\nCustomer segment performance")
segment_analysis

## 12. Measure the effect of discounts

In [ ]:
discount_analysis = sales.groupby("discount_pct").agg(
    orders=("order_id", "count"),
    units_sold=("units_sold", "sum"),
    revenue=("revenue", "sum"),
    profit=("profit", "sum"),
    average_profit=("profit", "mean"),
    return_rate_pct=("is_returned", lambda values: values.mean() * 100),
)
discount_analysis["profit_margin_pct"] = discount_analysis["profit"].div(discount_analysis["revenue"]).mul(100)
discount_analysis

## 13. Analyze returned orders

In [ ]:
return_analysis = sales.groupby(["category", "returned"]).agg(
    orders=("order_id", "count"),
    revenue=("revenue", "sum"),
    profit=("profit", "sum"),
).reset_index()
return_analysis.sort_values(["category", "returned"])

## 14. Build a region-by-category revenue table

In [ ]:
pd.pivot_table(
    sales,
    index="region",
    columns="category",
    values="revenue",
    aggfunc="sum",
    margins=True,
    margins_name="Total",
)

## 15. Find the most profitable orders

In [ ]:
sales.nlargest(10, "profit")[[
    "order_id", "order_date", "region", "product",
    "units_sold", "discount_pct", "revenue", "profit"
]]

## 16. Generate key findings

In [ ]:
findings = pd.DataFrame({
    "question": [
        "Highest-revenue category",
        "Highest-revenue region",
        "Highest-revenue channel",
        "Best-selling product by units",
        "Category with highest return rate",
        "Most profitable month",
    ],
    "answer": [
        category_analysis["revenue"].idxmax(),
        region_analysis["revenue"].idxmax(),
        channel_analysis["revenue"].idxmax(),
        product_analysis["units_sold"].idxmax()[1],
        category_analysis["return_rate_pct"].idxmax(),
        str(monthly_sales["profit"].idxmax()),
    ],
})
findings

## Practice questions

1. Which payment method produces the highest average order value?
2. Which product has the strongest profit margin?
3. Which region and category combination has the most returns?
4. Compare discounted and non-discounted orders.
5. Identify the three months with the highest revenue.